C:\Users\26959\AppData\Local\Temp\ipykernel_24012\2673749129.py:44: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[week_cols] = df[week_cols].applymap(clean_value)
C:\Users\26959\AppData\Local\Temp\ipykernel_24012\2673749129.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Region"] = cc.convert(names=df["Country"], to="UNregion")
C:\Users\26959\AppData\Local\Temp\ipykernel_24012\2673749129.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Subregion"] = cc.convert(names=df["C

In [23]:
# ---------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import country_converter as coco
import numpy as np
import time
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox
import re

# ---------------------------------------------------------------------
#  File paths
# ---------------------------------------------------------------------
file_path_main = "../results/Scale_up_output_MS.csv"
file_path_percent = "../results/Scale_up_PERCENT_ECW_POLL_MS.csv"
file_path_cr_man = "../results/Scale_up_CR_MAN_MS.csv"
file_path_cr_repur = "../results/Scale_up_CR_REPUR_MS.csv"
file_path_coalbag = "../results/Scale_up_COALBAG_MS.csv"
file_path_cr_stock = "../results/Scale_up_CR_STOCK.csv"
file_path_ecw_country = "../results/ECWbyCountry.csv"

# ---------------------------------------------------------------------
#  Load CSVs (main airflow + percentage + subtypes)
# ---------------------------------------------------------------------
def load_dataset(path, cumulative=True):
    df = pd.read_csv(path)
    df.rename(columns={df.columns[0]: "Country"}, inplace=True)
    week_cols = df.columns[3:]

    def clean_value(x):
        if isinstance(x, str):
            match = re.match(r"\(([\d\.]+)\+/-.*?\)(e[+-]?\d+)?", x)
            if match:
                base = match.group(1)
                exp = match.group(2) if match.group(2) else ""
                try: return float(base + exp)
                except: return np.nan
            match2 = re.match(r"([\d\.]+)\+/-", x)
            if match2:
                try: return float(match2.group(1))
                except: return np.nan
            try: return float(x)
            except: return np.nan
        return x

    df[week_cols] = df[week_cols].applymap(clean_value)
    if cumulative:
        df[week_cols] = df[week_cols].cumsum(axis=1)
    return df, week_cols

# Load datasets
df_main, week_cols = load_dataset(file_path_main, cumulative=True)
df_percent, _ = load_dataset(file_path_percent, cumulative=False)
df_cr_man, _ = load_dataset(file_path_cr_man, cumulative=True)
df_cr_repur, _ = load_dataset(file_path_cr_repur, cumulative=True)
df_coalbag, _ = load_dataset(file_path_coalbag, cumulative=True)
df_cr_stock, _ = load_dataset(file_path_cr_stock, cumulative=True)

# Load ECW lower/upper bounds
df_ecw = pd.read_csv(file_path_ecw_country)
df_ecw.rename(columns={df_ecw.columns[0]: "Country"}, inplace=True)
ecw_lower_dict = df_ecw.set_index("Country")["ECW ILO"].to_dict()
ecw_upper_dict = df_ecw.set_index("Country")["ECW Poll"].to_dict()

# ---------------------------------------------------------------------
#  Add region info
# ---------------------------------------------------------------------
cc = coco.CountryConverter()
def ensure_str(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return ", ".join(map(str, x))
    return str(x)

for frame in [df_main, df_percent, df_cr_man, df_cr_repur, df_coalbag, df_cr_stock]:
    frame["Region"] = pd.Series(cc.convert(names=frame["Country"], to="UNregion")).apply(ensure_str)
    frame["Subregion"] = pd.Series(cc.convert(names=frame["Country"], to="continent")).apply(ensure_str)
    frame["ISO3"] = pd.Series(cc.convert(names=frame["Country"], to="ISO3")).apply(ensure_str)

# ---------------------------------------------------------------------
#  Widgets
# ---------------------------------------------------------------------
map_choice = widgets.Dropdown(
    options=["Total Air Flow", "ECW Coverage %"],
    value="Total Air Flow", description="Map:",
    style={'description_width': 'initial'}
)
type_choice = widgets.Dropdown(
    options=["ALL", "CR_MAN", "CR_REPUR", "COALBAG", "CR_STOCK"],
    value="ALL", description="Type:",
    style={'description_width': 'initial'}
)
region_filter = widgets.Dropdown(
    options=["All"]
    + sorted(df_main["Region"].dropna().unique().tolist())
    + sorted(df_main["Subregion"].dropna().unique().tolist()),
    value="All", description="UN Region:", style={'description_width': 'initial'}
)
region_select = widgets.SelectMultiple(
    options=sorted(df_main["Region"].dropna().unique().tolist()),
    value=["Eastern Asia"], description="Regions:",
    layout=widgets.Layout(width="50%", height="150px")
)
week_index = widgets.IntSlider(
    value=1, min=1, max=len(week_cols), step=1, description="Week:",
    layout=widgets.Layout(width='80%')
)
play_button = widgets.Button(
    description="▶ Play", tooltip="Auto-play weeks",
    button_style="success", layout=widgets.Layout(width='100px', height='35px')
)
out = widgets.Output()

# ---------------------------------------------------------------------
#  Plot update function
# ---------------------------------------------------------------------
def update_plots(change=None):
    week = str(week_index.value)
    map_val = map_choice.value
    type_val = type_choice.value
    filter_value = region_filter.value

    with out:
        clear_output(wait=True)

        # ---------- Choropleth (first map)
        if map_val == "Total Air Flow":
            df_map = df_main.copy()
            vmin, vmax = df_map[week_cols].min().min(), df_map[week_cols].max().max()
            color_scale = [(0, "white"), (1, "darkgreen")]
            title_text = f"Total Air Flow by Country (up to Week {week})"
        else:
            df_map = df_percent.copy()
            vmin, vmax = 0, 1
            color_scale = [(0, "white"), (1, "orange")]
            title_text = f"ECW Coverage (%) by Country (Week {week})"

        if filter_value != "All":
            df_map = df_map[(df_map["Region"] == filter_value) | (df_map["Subregion"] == filter_value)]

        fig1 = px.choropleth(
            df_map, locations="ISO3", locationmode="ISO-3",
            color=week, hover_name="Country",
            color_continuous_scale=color_scale,
            range_color=[vmin, vmax], title=title_text
        )
        fig1.update_layout(margin=dict(l=0, r=0, t=50, b=0))
        fig1.show()

        # --- Week slider + play button
        control_row = HBox([week_index, play_button],
                           layout=widgets.Layout(justify_content="center", align_items="center", margin="10px 0px"))
        display(control_row)

        # ---------- Second chart: Airflow with ECW range (based on Type)
        fig, ax = plt.subplots(figsize=(10,6))
        if type_val == "ALL": df_used = df_main
        elif type_val == "CR_MAN": df_used = df_cr_man
        elif type_val == "CR_REPUR": df_used = df_cr_repur
        elif type_val == "COALBAG": df_used = df_coalbag
        elif type_val == "CR_STOCK": df_used = df_cr_stock
        else: df_used = df_main

        for region in region_select.value:
            subdf = df_used[df_used["Region"] == region]
            if subdf.empty: continue
            mean_series = subdf[week_cols].mean()
            t = np.arange(len(mean_series))
            ax.plot(t, mean_series, label=f"{region} ({type_val})", linewidth=2)

            # --- ECW Bounds from ECWbyCountry.csv ---
            countries = subdf["Country"].unique()
            ecw_lowers = [ecw_lower_dict.get(c, np.nan) for c in countries]
            ecw_uppers = [ecw_upper_dict.get(c, np.nan) for c in countries]
            lower = np.nanmean(ecw_lowers)
            upper = np.nanmean(ecw_uppers)
            ax.axhspan(lower, upper, color='green', alpha=0.15)

        ax.set_title(f"{type_val} Air Flow vs ECW Range by Selected UN Regions")
        ax.set_xlabel("Weeks")
        ax.set_ylabel("Air Flow")
        ax.legend(loc="upper left", bbox_to_anchor=(1,1))
        plt.show()

# ---------------------------------------------------------------------
#  Play logic
# ---------------------------------------------------------------------
def play_clicked(b):
    for w in range(1, len(week_cols) + 1):
        week_index.value = w
        time.sleep(0.5)

play_button.on_click(play_clicked)
week_index.observe(update_plots, names="value")
region_filter.observe(update_plots, names="value")
region_select.observe(update_plots, names="value")
map_choice.observe(update_plots, names="value")
type_choice.observe(update_plots, names="value")

# ---------------------------------------------------------------------
#  Style + Layout
# ---------------------------------------------------------------------
custom_style = """
<style>
#custom-container {
    background-color: #fffacd;
    padding: 15px;
    border-radius: 10px;
    margin-top: 10px;
    margin-bottom: 20px;
}
#custom-banner {
    width: 100%;
    background-color: #006400;
    color: white;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    padding: 10px;
    border-radius: 6px;
    margin-bottom: 15px;
}
</style>
"""
display(HTML(custom_style))

container = VBox([
    widgets.HTML('<div id="custom-banner">ALLFED</div>'),
    HBox([region_filter, map_choice, type_choice]),
    region_select,
    out
])
container.layout = widgets.Layout(
    width="100%", padding="15px",
    border="solid 2px #006400", border_radius="10px",
    background_color="#fffacd"
)
display(container)
update_plots()


C:\Users\26959\AppData\Local\Temp\ipykernel_6392\3790149824.py:50: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\26959\AppData\Local\Temp\ipykernel_6392\3790149824.py:50: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\26959\AppData\Local\Temp\ipykernel_6392\3790149824.py:50: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\26959\AppData\Local\Temp\ipykernel_6392\3790149824.py:50: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\26959\AppData\Local\Temp\ipykernel_6392\3790149824.py:50: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

C:\Users\26959\AppData\Local\Temp\ipykernel_6392\3790149824.py:50: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

More than one regular expression match for Australia and New Zealand
More than one regular exp